# Assignment 3: Fine-tuning language models

In this assignment, you will perform supervised fine-tuning (SFT) of a small open LLM on an instruction tuning dataset. You will convert this dataset into instruction-response pairs, fine-tune a causal language model using LoRA (Low-Rank Adaptation), and evaluate it through prompted inference and comparison with other methods.

## Preliminaries

First, let's install the required libraries. If you are running in your own environment, make sure the following are installed:

- [Torch](https://docs.pytorch.org/docs/stable/index.html)
- [Transformers](https://huggingface.co/docs/transformers/index)
- [Datasets](https://huggingface.co/docs/datasets/index)
- [Evaluate](https://huggingface.co/docs/evaluate/en/index)
- [NLTK](https://www.nltk.org/api/nltk.html)
- [rouge_score](https://pypi.org/project/rouge-score/)

In a Colab notebook, most of them are already installed, except Evaluate and rouge_score.

## Local Apple Silicon / MPS setup

This copy is adapted for running locally on an Apple Silicon Mac.

Main changes:
- install the full local Python dependency set;
- prefer Apple's `mps` backend, then CUDA, then CPU;
- enable CPU fallback for PyTorch operations not implemented on MPS;
- choose bf16/fp16 settings according to the available backend;
- replace hard-coded `.to("cuda")` calls with the detected device;
- use local output directories for HuggingFace `Trainer`.

The assignment TODOs themselves are intentionally left unsolved.


In [ ]:
# Recommended: run this notebook inside a fresh Python virtual environment.
# `%pip` installs into the Python environment used by this Jupyter kernel.
%pip install torch transformers datasets evaluate nltk rouge_score accelerate


We also set some configuration parameters.

Most importantly, you should select a language model to work with in this assignment and enter its HuggingFace identifier in the parameter `MODEL_NAME` below. In principle you can use any model that you want, but we recommend that you select a model that has not already been trained to follow instructions, so it should be a "pure" language model trained on raw text (similar to Assignments 1 and 2).

The selected model should be small enough to fit in your computational environment. We have verified that the 135-million parameter [`SmolLM2` model](https://huggingface.co/HuggingFaceTB/SmolLM2-135M), developed by HuggingFace, can be used to solve this assignment in a Colab notebook (free tier, T4 GPU). If you run on a cluster, you can select a larger model (and probably see more interesting results).

We also define training and test set sizes here. Again, the values below have been set so that the assignment can be solved in Colab, and you can increase these sizes to improve the quality of the fine-tuned models.

In [1]:
# Set this before importing torch so unsupported MPS ops can fall back to CPU.
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import json
import platform
from pathlib import Path
import torch

SEED = 101
torch.manual_seed(SEED)

# Prefer Apple Silicon GPU (MPS), then NVIDIA CUDA, then CPU.
if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

# Mixed precision settings suitable for the detected backend.
# Modern Apple Silicon/macOS supports bf16; CUDA uses bf16 when the GPU supports it,
# otherwise fp16. CPU uses fp32 here for simplicity.
if DEVICE == "mps":
    mac_major = int(platform.mac_ver()[0].split(".")[0]) if platform.mac_ver()[0] else 0
    USE_BF16 = mac_major >= 14
    USE_FP16 = not USE_BF16
elif DEVICE == "cuda":
    USE_BF16 = torch.cuda.is_bf16_supported()
    USE_FP16 = not USE_BF16
else:
    USE_BF16 = False
    USE_FP16 = False

# Pinned host memory is mainly useful for CUDA transfers, not MPS.
PIN_MEMORY = (DEVICE == "cuda")

MAX_TRAIN_SAMPLES = 5000
MAX_TEST_SAMPLES = 400

MODEL_NAME = "HuggingFaceTB/SmolLM2-135M"

OUTPUT_ROOT = Path("./a3_outputs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
print("bf16:", USE_BF16, "| fp16:", USE_FP16)
print("Output directory:", OUTPUT_ROOT.resolve())


PyTorch: 2.12.0
Device: mps
bf16: True | fp16: False
Output directory: /Users/siyaoliu/dl4nlp-assignments/a3/a3_outputs


In [2]:
# Quick local hardware sanity check
x = torch.ones(4, device=DEVICE)
print("Tensor device:", x.device)
print("MPS built:", torch.backends.mps.is_built())
print("MPS available:", torch.backends.mps.is_available())


Tensor device: mps:0
MPS built: True
MPS available: True


# Part 1: Preprocessing

### ⚙&nbsp; Task 1.1: Loading and inspecting the dataset

The dataset [SmolTalk](https://huggingface.co/datasets/HuggingFaceTB/smoltalk) is a collection of instruction-response pairs designed for SFT of large language models for instruction following. This dataset consists of examples of user inputs with system responses.

You can load using the datasets from the HuggingFace repository as follows.

In [3]:
from datasets import load_dataset
from datasets import DatasetDict

smoltalk = load_dataset("HuggingFaceTB/smoltalk", 'all')

/Users/siyaoliu/dl4nlp-assignments/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating test split: 100%|██████████| 54948/54948 [00:00<00:00, 455127.98 examples/s]


In order to make this assignment possible to solve in a restricted environment, we simplify the dataset a bit:
- We remove multi-turn chat dialogues from the dataset;
- We remove instances where the query or the answer is greater than a set maximum length;
- We keep a subset of the data for training and testing (by default 5000 and 400, respectively).

In [4]:
smoltalk_simplified = smoltalk.filter(lambda row: len(row['messages']) <= 3 and all(len(m['content']) <= 256 for m in row['messages']))
smoltalk_simplified = DatasetDict({
    "train": smoltalk_simplified["train"].select(range(MAX_TRAIN_SAMPLES)),
    "test": smoltalk_simplified["test"].select(range(MAX_TEST_SAMPLES)),
})

Filter: 100%|██████████| 54948/54948 [00:00<00:00, 93994.74 examples/s] 


In [5]:
smoltalk_simplified

DatasetDict({
    train: Dataset({
        features: ['messages', 'source'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['messages', 'source'],
        num_rows: 400
    })
})

Print some examples from the dataset so that you understand the format.

Key points you need to note here: each example from the training or test set consists of a sequence of messages. The number of messages in each example will be 2 or 3, because we removed multi-turn chat dialogues in the previous step. Each message is associated with a `role` label:
- `user`: an example of something the user might write.
- `assistant`: an example of an output an LLM could be expected to produce, given the input.
- `system`: a *system prompt* that gives guidelines for the general behavior of the LLM's behavior.

All examples in the dataset include a user input and an assistant output, but the system prompt is not available in all of the examples.

In [6]:
smoltalk_simplified['train'][0]

{'messages': [{'content': "You are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.",
   'role': 'system'},
  {'content': 'Rearrange this sentence to make it easy to understand:\nThe restaurant ran out of food, so the chef made some more.',
   'role': 'user'},
  {'content': 'The chef made more food after the restaurant ran out.',
   'role': 'assistant'}],
 'source': 'explore-instruct-rewriting'}

### 🎓&nbsp; Task 1.2: Formatting the data for instruction tuning

Define a function `format_input_output` that converts an example from the dataset into an input/output pair that we can use to fine-tune the LLM.

You are free to design the format. The following document gives some examples that have been used by different instruction-following LLMs including Llama and Mistral: https://huggingface.co/learn/llm-course/chapter11/2#common-template-formats

The later stages of our preprocessing pipeline expect that this function returns an object containing two parts: the `prompt` (what goes into the LLM before generating anything) and the `response` (what the LLM is expected to generate).

In [7]:
def format_input_output(example):
  # `messages` is a list of messages, each with a `content` string and a `role`.
  messages = example['messages']

  # TODO: implement this

  systems = [m["content"] for m in messages if m["role"] == "system"]
  users = [m["content"] for m in messages if m["role"] == "user"]
  assistants = [m["content"] for m in messages if m["role"] == "assistant"]

  assert len(systems) <= 1
  assert len(users) == 1
  assert len(assistants) == 1

  if systems:
      prompt = (
          f"System: {systems[0]}\n"
          f"User: {users[0]}\n"
          f"Assistant:"
      )
  else:
      prompt = (
          f"User: {users[0]}\n"
          f"Assistant:"
      )

  return {
      "prompt": prompt,
      "response": f" {assistants[0]}",
  }

Apply the function you implemented to the dataset as a whole.

In [8]:
ds_sft = smoltalk_simplified.map(format_input_output)

Map: 100%|██████████| 400/400 [00:00<00:00, 9661.85 examples/s]


Then verify that the dataset now contains the new fields you created.

In [9]:
ds_sft['train'][0]

{'messages': [{'content': "You are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.",
   'role': 'system'},
  {'content': 'Rearrange this sentence to make it easy to understand:\nThe restaurant ran out of food, so the chef made some more.',
   'role': 'user'},
  {'content': 'The chef made more food after the restaurant ran out.',
   'role': 'assistant'}],
 'source': 'explore-instruct-rewriting',
 'prompt': "System: You are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.\nUser: Rearrange this sentence to make it easy to understand:\nThe restaurant ran out of food, so the chef made some more.\nAssistant:",
 'response': ' The chef made more food after the restaurant ran out.'}

### ⚙&nbsp; Task 1.3: Tokenizing the dataset

We will now prepare the format required by the HuggingFace Trainer.

We first load the tokenizer for our selected model:

In [16]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [17]:
print(tokenizer)
print("pad token:", tokenizer.pad_token, tokenizer.pad_token_id)
print("eos token:", tokenizer.eos_token, tokenizer.eos_token_id)

GPT2Tokenizer(name_or_path='HuggingFaceTB/SmolLM2-135M', vocab_size=49152, model_max_length=8192, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	0: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<repo_name>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("<reponame>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	5: AddedToken("<file_sep>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	6: AddedToken("<filename>", rstrip

Write a function `tokenize_helper` that takes an example (using the prompt/response format from the previous step) and produces the following three results:

- `input_ids`: the integer token ids of the concatenated prompt and response;
- `labels`: a list of the same length as `input_ids`, where the response token ids are the same, but where the prompt token ids have all been replaced by the loss masking identifier -100.
- `attention_mask`: the attention mask. This should just be a list of the same length as the other two lists, with all items set to 1.

The reason why `input_ids` and `labels` are different is that
we do not want to compute the training loss for tokens that appear in the user's input. We want to train the model to generate output *conditionally*: based on a prompt. But why the magic number -100? This is the number used by default in PyTorch's [`CrossEntropyLoss`](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) to indicate an item that should be excluded in loss computations. (This issue was also mentioned in [Assignment 1](https://liu-nlp.ai/dl4nlp/units/a1_1.html#task-4.1-implementing-the-trainer).)

In [44]:
def tokenize_helper(example):
    prompt = example['prompt']     # Created in the previous step
    response = example['response'] # Created in the previous step

    # TODO: your work goes here.

    prompt_ids = tokenizer.encode(
        prompt,
        add_special_tokens=False,
    )

    response_ids = tokenizer.encode(
        response,
        add_special_tokens=False,
    )

    response_ids = response_ids + [tokenizer.eos_token_id]

    input_ids = prompt_ids + response_ids

    labels = (
        [-100] * len(prompt_ids)
        + response_ids
    )

    attention_mask = [1] * len(input_ids)

    return {
        "input_ids": input_ids,       # Input token ids of the prompt and response
        "attention_mask": attention_mask,  # Attention mask of the prompt and response
        "labels": labels          # Output token ids of the prompt (masked) and response
    }


As above, apply the function you implemented to the dataset using `map`. This will add the three new fields to the dataset.

In [45]:
example = ds_sft["train"][0]

tokenized = tokenize_helper(example)

print("input length:", len(tokenized["input_ids"]))
print("label length:", len(tokenized["labels"]))
print("mask length:", len(tokenized["attention_mask"]))

input length: 76
label length: 76
mask length: 76


In [14]:
first_response_idx = tokenized["labels"].index(
    next(x for x in tokenized["labels"] if x != -100)
)

print("First response index:", first_response_idx)

print("\nPrompt decoded:")
print(
    tokenizer.decode(
        tokenized["input_ids"][:first_response_idx]
    )
)

print("\nResponse decoded:")
print(
    tokenizer.decode(
        tokenized["input_ids"][first_response_idx:]
    )
)

First response index: 64

Prompt decoded:
System: You are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.
User: Rearrange this sentence to make it easy to understand:
The restaurant ran out of food, so the chef made some more.
Assistant:

Response decoded:
 The chef made more food after the restaurant ran out.


In [15]:
tokenized_ds_sft = ds_sft.map(tokenize_helper)
tokenized_ds_sft["train"][0]

Map: 100%|██████████| 400/400 [00:00<00:00, 7870.12 examples/s]


{'messages': [{'content': "You are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.",
   'role': 'system'},
  {'content': 'Rearrange this sentence to make it easy to understand:\nThe restaurant ran out of food, so the chef made some more.',
   'role': 'user'},
  {'content': 'The chef made more food after the restaurant ran out.',
   'role': 'assistant'}],
 'source': 'explore-instruct-rewriting',
 'prompt': "System: You are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.\nUser: Rearrange this sentence to make it easy to understand:\nThe restaurant ran out of food, so the chef made some more.\nAssistant:",
 'response': ' The chef made more food after the restaurant ran out.',
 'input_ids': [18403,
  42,
  1206,
  359,
  354,
  5646,
  298,
  12021,
  11173,
  30,
  1206,
  523,
  325,
  2711,
  351,
  253,
  1694,
  284,
  346,
  737,
  


## Part 2: Evaluation of the baseline model

As a first step, we will see how well the *baseline* model performs: that is, a model that has not been trained to follow instructions.

### ⚙&nbsp; Task 2.1: Preparing for evaluation

In this section, we set up a few utilities we will need to complete our training and evaluation infrastructure. These utilities will be given and you don't need to modify anything.

The first piece we need is a *collator*: that is, a tool that takes a number of instances and creates PyTorch tensors for a training batch. To make the batch fit into rectangular tensors, padding tokens will be added.

In [19]:
def data_collator(batch):
    """
    Create a custom collate function for causal language modeling.

    Args:
        batch: List of examples, each with 'input_ids', 'attention_mask', 'labels'
        tokenizer: Tokenizer with pad_token_id
    """

    input_ids_list = [torch.tensor(example["input_ids"], dtype=torch.long) for example in batch]
    attention_masks_list = [torch.tensor(example["attention_mask"], dtype=torch.long) for example in batch]
    labels_list = [torch.tensor(example['labels'], dtype=torch.long) for example in batch]

    # Find max length in this batch
    max_len = max(x.size(0) for x in input_ids_list)

    # Helper pad function
    def pad_to_max(x_list, pad_value):
        padded = []
        for x in x_list:
            pad_len = max_len - x.size(0)
            if pad_len > 0:
                pad_tensor = torch.full((pad_len,), pad_value, dtype=x.dtype)
                x = torch.cat([x, pad_tensor], dim=0)
            padded.append(x)
        return torch.stack(padded, dim=0)

    # Use tokenizer.pad_token_id for inputs, 0 for attention_mask, -100 for labels
    pad_id = tokenizer.pad_token_id

    batch_input_ids = pad_to_max(input_ids_list, pad_value=pad_id)
    batch_attention_mask = pad_to_max(attention_masks_list, pad_value=0)
    batch_labels = pad_to_max(labels_list, pad_value=-100)

    batch = {
            "input_ids": batch_input_ids,
            "attention_mask": batch_attention_mask,
            "labels": batch_labels,
        }
    return batch

The second utility we need is an evaluator. We will use the **ROUGE-L** metric, which computes the longest common subsequence between the model's output and the gold-standard answer. You can read about ROUGE-L here: https://en.wikipedia.org/wiki/ROUGE_(metric)

When using the ROUGE-L metric in a Trainer, we need to wrap it in an object defined as follows:

In [20]:
import evaluate

class RougeMetricComputer:
    """
    Stateful metric for batch_eval_metrics=True.

    It:
      - accumulates predictions and references across batches
      - computes ROUGE-L once at the end (compute_result=True)
    """

    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.rouge = evaluate.load("rouge")
        self.all_predictions = []
        self.all_references = []

    def __call__(self, eval_pred, compute_result=False):
        """Accumulate predictions and compute at the end."""

        logits, labels = eval_pred
        pred_ids = logits.argmax(axis=-1)

        # Collect decoded answer-span text from each example in the batch
        for p, lbl in zip(pred_ids, labels):
            mask = lbl != -100
            if mask.sum() == 0:
                continue

            ref_ids = lbl[mask]
            pred_ids_filtered = p[mask]

            ref_text = self.tokenizer.decode(ref_ids, skip_special_tokens=True)
            pred_text = self.tokenizer.decode(
                pred_ids_filtered, skip_special_tokens=True,
                eos_token_id=self.tokenizer.vocab['<|im_end|>']
            )

            self.all_references.append(ref_text.strip())
            self.all_predictions.append(pred_text.strip())

        # Only compute at the very end of eval
        if compute_result:
            if len(self.all_references) > 0:
                scores = self.rouge.compute(
                    predictions=self.all_predictions,
                    references=self.all_references,
                )

                # Clear accumulated data for next eval call
                self.all_predictions = []
                self.all_references = []
                return {"rougeL": scores["rougeL"]}
            else:
                return {}
        else:
            return {}

compute_metrics = RougeMetricComputer(tokenizer)


Finally, we make a function that sets up a [`Trainer`](https://huggingface.co/docs/transformers/main_classes/trainer).

In [21]:
from transformers import Trainer
from transformers.trainer_callback import ProgressCallback

def make_trainer(model, training_args):
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_ds_sft["train"],
        eval_dataset=tokenized_ds_sft["test"],
        compute_metrics=compute_metrics,
        data_collator=data_collator,
    )
    trainer.callback_handler.callbacks = [
        cb for cb in trainer.callback_handler.callbacks
        if type(cb).__name__ != "NotebookProgressCallback"
    ]
    trainer.add_callback(ProgressCallback)
    return trainer


### 🎓&nbsp; Task 2.2: Evaluating the pre-trained model

Now, we have all the pieces to evaluate our baseline model that has not been instruction-tuned.

The following code will compute the loss on the test set as well as the ROUGE-L score. You will later compare these scores to the models that you train.

Why do you think the ROUGE-L score is as high as it is, even without any training for instruction-following?

In [22]:
from transformers import TrainingArguments
from transformers import AutoModelForCausalLM
import time

print("\n" + "=" * 80)
print("EVALUATING PRETRAINED MODEL")
print("=" * 80)

# Keep the assignment's base model; use the detected Mac MPS/CUDA/CPU device.
pretrained_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)

pretrained_eval_args = TrainingArguments(
    output_dir=str(OUTPUT_ROOT / "pretrained_eval"),
    eval_strategy="no",
    per_device_eval_batch_size=1,
    bf16=USE_BF16,
    fp16=USE_FP16,
    report_to="none",
    batch_eval_metrics=True,
    eval_accumulation_steps=1,
    dataloader_pin_memory=PIN_MEMORY,
)

pretrained_trainer = make_trainer(pretrained_model, pretrained_eval_args)

t0 = time.perf_counter()
pretrained_eval_metrics = pretrained_trainer.evaluate()
pretrained_eval_time = time.perf_counter() - t0

pretrained_eval_loss = float(pretrained_eval_metrics["eval_loss"])
pretrained_rougeL = pretrained_eval_metrics.get("eval_rougeL", None)

print("\nPRETRAINED EVAL METRICS:")
print(json.dumps(pretrained_eval_metrics, indent=2))
print(f"Evaluation time: {pretrained_eval_time:.1f}s")



EVALUATING PRETRAINED MODEL


100%|██████████| 400/400 [00:19<00:00, 20.13it/s]


PRETRAINED EVAL METRICS:
{
  "eval_loss": 1.7945466041564941,
  "eval_model_preparation_time": 0.0019,
  "eval_rougeL": 0.5957066566538843,
  "eval_runtime": 21.3776,
  "eval_samples_per_second": 18.711,
  "eval_steps_per_second": 18.711,
  "epoch": 0
}
Evaluation time: 21.4s



## Part 3: Supervised fine-tuning



### 🎓&nbsp; Task 3.1: Training the full model

Next, we train the pre-trained model using SFT over all the parameters, then calculate the metrics and outputs to evaluate how well it follows instructions.

How do the results differ from those in the previous step?

In [23]:
baseline_training_args = TrainingArguments(
    output_dir=str(OUTPUT_ROOT / "full_sft"),
    eval_strategy="epoch",
    logging_steps=2000,
    save_strategy="no",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    bf16=USE_BF16,
    fp16=USE_FP16,
    report_to="none",
    batch_eval_metrics=True,
    eval_accumulation_steps=1,
    dataloader_pin_memory=PIN_MEMORY,
)

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)

baseline_trainer = make_trainer(base_model, baseline_training_args)

# TODO: train and evaluate the model.

import time

t0 = time.perf_counter()

train_result = baseline_trainer.train()

full_sft_train_time = time.perf_counter() - t0

full_sft_eval_metrics = baseline_trainer.evaluate()

full_sft_eval_loss = float(
    full_sft_eval_metrics["eval_loss"]
)

full_sft_rougeL = full_sft_eval_metrics.get(
    "eval_rougeL",
    None
)

print("\nFULL SFT EVAL METRICS:")
print(json.dumps(full_sft_eval_metrics, indent=2))

print(f"\nTraining time: {full_sft_train_time:.1f}s")


 40%|████      | 2001/5000 [12:41<20:27,  2.44it/s]

{'loss': '1.493', 'grad_norm': '5.125', 'learning_rate': '3.001e-05', 'epoch': '0.4'}


 80%|████████  | 4001/5000 [24:58<06:16,  2.66it/s]

{'loss': '1.363', 'grad_norm': '9', 'learning_rate': '1.001e-05', 'epoch': '0.8'}


                                                   
100%|██████████| 5000/5000 [31:18<00:00,  2.66it/s]


{'eval_loss': '1.39', 'eval_rougeL': '0.6391', 'eval_runtime': '13.2', 'eval_samples_per_second': '30.31', 'eval_steps_per_second': '30.31', 'epoch': '1'}
{'train_runtime': '1878', 'train_samples_per_second': '2.662', 'train_steps_per_second': '2.662', 'train_loss': '1.409', 'epoch': '1'}


100%|██████████| 400/400 [00:12<00:00, 32.55it/s]


FULL SFT EVAL METRICS:
{
  "eval_loss": 1.3904308080673218,
  "eval_rougeL": 0.6391446634575103,
  "eval_runtime": 12.3142,
  "eval_samples_per_second": 32.483,
  "eval_steps_per_second": 32.483,
  "epoch": 1.0
}

Training time: 1878.4s


### ⚙&nbsp; Task 3.3: Counting the number of trainable parameters

Define a function `num_trainable_parameters` that computes the number of floating-point numbers that a given model will update during training.

**Hints**:
- For a PyTorch module `m`, you can use `m.parameters()` to access its parameter tensors.
- However, you should only include parameter tensors where the flag `requires_grad` is True.


In [25]:
def num_trainable_parameters(model):
    """Count number of trainable parameters.

    Args:
        model: A PyTorch module.
    """
    # TODO: Add your code here
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

Apply this function to the SFT-trained model and check that the result makes sense.

In [26]:
full_sft_trainable_params = num_trainable_parameters(base_model)

print("Full SFT trainable parameters:", full_sft_trainable_params)
print(f"{full_sft_trainable_params / 1e6:.2f}M")

Full SFT trainable parameters: 134515008
134.52M


In [27]:
total_params = sum(
    p.numel()
    for p in base_model.parameters()
)

print("Total parameters:", total_params)
print("Trainable parameters:", full_sft_trainable_params)
print(
    "Trainable percentage:",
    100 * full_sft_trainable_params / total_params
)

Total parameters: 134515008
Trainable parameters: 134515008
Trainable percentage: 100.0


## Part 4: Parameter-efficient fine-tuning

In the last section of this assignment, we will use LoRA to train the model in a more parameter-efficient manner. You may want to prepare by reading  by [the paper by Hu et al. (2021)](https://arxiv.org/pdf/2106.09685) and the teaching material provided for this course.

### ⚙&nbsp; Task 4.1: Utilities for modifying models

Define a function `extract_lora_targets` that extracts the relevant linear layers from all Transformer blocks in your selected LLM.
It is up to you to decide what layers to select; in the experiments described in the original LoRA paper, the query and value projection matrices were fine-tuned with LoRA, while all other layers were left unchanged.
Return a dictionary that maps the component name to the corresponding linear layer.

As we saw earlier (in Assignment 2 and elsewhere), a Transformer model consists of a hierarchy of nested submodules. Each of these can be addressed by a fully-qualified string name. You can use get_submodule() to retrieve a layer by a string name. This name depends on the model you have selected. For instance, in the `SmolLM2-135M` model, `'model.layers.0.self_attn.q_proj'`
 refers to the query projection in Transformer layer 0.

It is OK to hard-code this part, so that you just enumerate the layers you want to extract. Alternatively, use a utility such as `model.named_modules()` to iterate through the model's layers.

In [30]:
import torch.nn as nn
for name, module in base_model.named_modules():
    if "self_attn" in name:
        print(name, type(module))

model.layers.0.self_attn <class 'transformers.models.llama.modeling_llama.LlamaAttention'>
model.layers.0.self_attn.q_proj <class 'torch.nn.modules.linear.Linear'>
model.layers.0.self_attn.k_proj <class 'torch.nn.modules.linear.Linear'>
model.layers.0.self_attn.v_proj <class 'torch.nn.modules.linear.Linear'>
model.layers.0.self_attn.o_proj <class 'torch.nn.modules.linear.Linear'>
model.layers.1.self_attn <class 'transformers.models.llama.modeling_llama.LlamaAttention'>
model.layers.1.self_attn.q_proj <class 'torch.nn.modules.linear.Linear'>
model.layers.1.self_attn.k_proj <class 'torch.nn.modules.linear.Linear'>
model.layers.1.self_attn.v_proj <class 'torch.nn.modules.linear.Linear'>
model.layers.1.self_attn.o_proj <class 'torch.nn.modules.linear.Linear'>
model.layers.2.self_attn <class 'transformers.models.llama.modeling_llama.LlamaAttention'>
model.layers.2.self_attn.q_proj <class 'torch.nn.modules.linear.Linear'>
model.layers.2.self_attn.k_proj <class 'torch.nn.modules.linear.Linear

In [31]:
def extract_lora_targets(model):
  # TODO: Add your code here
  targets = {}

  target_names = ("q_proj", "k_proj", "v_proj", "o_proj")

  for name, module in model.named_modules():
      if (
          isinstance(module, nn.Linear)
          and name.endswith(target_names)
          and ".self_attn." in name
      ):
          targets[name] = module

  return targets

In [32]:
targets = extract_lora_targets(base_model)

print("Number of target layers:", len(targets))

for name, layer in list(targets.items())[:8]:
    print(name, layer)

Number of target layers: 120
model.layers.0.self_attn.q_proj Linear(in_features=576, out_features=576, bias=False)
model.layers.0.self_attn.k_proj Linear(in_features=576, out_features=192, bias=False)
model.layers.0.self_attn.v_proj Linear(in_features=576, out_features=192, bias=False)
model.layers.0.self_attn.o_proj Linear(in_features=576, out_features=576, bias=False)
model.layers.1.self_attn.q_proj Linear(in_features=576, out_features=576, bias=False)
model.layers.1.self_attn.k_proj Linear(in_features=576, out_features=192, bias=False)
model.layers.1.self_attn.v_proj Linear(in_features=576, out_features=192, bias=False)
model.layers.1.self_attn.o_proj Linear(in_features=576, out_features=576, bias=False)


We also need a convenience function that puts layers back into a model. The following function does the trick. The `named_layers` argument uses the same format as returned by `extract_lora_targets`.

In [33]:
def replace_layers(model, named_layers):
    """
    Replace submodules in `model` by name.
    """
    for name, layer in named_layers.items():
        components = name.split(".")
        submodule = model
        for comp in components[:-1]:
            submodule = getattr(submodule, comp)
        setattr(submodule, components[-1], layer)
    return model

### 🎓&nbsp; Task 4.2: Implementing the LoRA layer

To implement the LoRA approach, we define a new type of layer that will be used as a drop-in replacement for a regular linear layer.

In [the paper by Hu et al. (2021)](https://arxiv.org/pdf/2106.09685), the structure is presented visually in Figure 1, and equation (3) shows the same idea.

Start from the following skeleton and fill in the missing pieces:


In [34]:
import torch.nn as nn

class LoRALayer(nn.Module):
    def __init__(self, W, r, alpha):
        super().__init__()
        # TODO: Add your code here
        self.W = W
        self.r = r
        self.alpha = alpha

        # Freeze the original pretrained linear layer
        for param in self.W.parameters():
            param.requires_grad = False

        # Low-rank adaptation matrices
        self.A = nn.Linear(
            W.in_features,
            r,
            bias=False
        )

        self.B = nn.Linear(
            r,
            W.out_features,
            bias=False
        )

        # Start with zero LoRA contribution
        nn.init.normal_(self.A.weight, std=0.02)
        nn.init.zeros_(self.B.weight)

    def forward(self, x):
        # TODO: Add your code here
        original_output = self.W(x)
        lora_output = self.B(self.A(x))

        return original_output + (self.alpha / self.r) * lora_output

Here, `W` is the linear layer we are fine-tuning, while `r` and `alpha` are hyperparameters described in section 4.1. of the paper. The `r` parameter controls the parameter efficiency: by setting it to a low value, we save memory but make a rougher approximation. The `alpha` parameter is a scaling factor.

In [35]:
W = targets["model.layers.0.self_attn.q_proj"]

lora_test = LoRALayer(W, r=8, alpha=16)

print(lora_test)
print(
    "Trainable params:",
    num_trainable_parameters(lora_test)
)

LoRALayer(
  (W): Linear(in_features=576, out_features=576, bias=False)
  (A): Linear(in_features=576, out_features=8, bias=False)
  (B): Linear(in_features=8, out_features=576, bias=False)
)
Trainable params: 9216


### 🎓&nbsp; Task 4.3: Fine-tuning with LoRA

Set up a model where you replace the four linear layers in attention blocks (query, key, value, and output) with LoRA layers. Use the following steps:
- First use `extract_lora_targets` to get the relevant linear layers.
- Each of the linear layers in the returned dictionary should be wrapped inside a LoRA layer.
- Then use `replace_layers` to put them back into the model.

Train this model and compare the training speed, metrics, and outputs to the results from Part 3.

Apply your parameter counting function (`num_trainable_parameters`) to this model, compare the results to those in Part 3, and make sure that these results correspond to your expectations.


In [36]:
from transformers import AutoModelForCausalLM

lora_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

for param in lora_model.parameters():
    param.requires_grad = False

print(num_trainable_parameters(lora_model))

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 5751.73it/s]


0


In [37]:
targets = extract_lora_targets(lora_model)

print("LoRA target layers:", len(targets))

LoRA target layers: 120


In [38]:
R = 8
ALPHA = 16

lora_layers = {
    name: LoRALayer(
        W=layer,
        r=R,
        alpha=ALPHA,
    )
    for name, layer in targets.items()
}

lora_model = replace_layers(
    lora_model,
    lora_layers,
)

lora_model = lora_model.to(DEVICE)

In [39]:
lora_trainable_params = num_trainable_parameters(lora_model)

total_lora_params = sum(
    p.numel()
    for p in lora_model.parameters()
)

print("LoRA trainable parameters:", lora_trainable_params)
print(f"LoRA trainable parameters: {lora_trainable_params / 1e6:.2f}M")

print(
    "Trainable percentage:",
    100 * lora_trainable_params / total_lora_params
)

LoRA trainable parameters: 921600
LoRA trainable parameters: 0.92M
Trainable percentage: 0.6804659490586179


In [40]:
from transformers import TrainingArguments

lora_training_args = TrainingArguments(
    output_dir=str(OUTPUT_ROOT / "lora_sft"),
    eval_strategy="epoch",
    logging_steps=2000,
    save_strategy="no",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    bf16=USE_BF16,
    fp16=USE_FP16,
    report_to="none",
    batch_eval_metrics=True,
    eval_accumulation_steps=1,
    dataloader_pin_memory=PIN_MEMORY,
)

lora_trainer = make_trainer(
    lora_model,
    lora_training_args,
)

In [41]:
import time

t0 = time.perf_counter()

lora_train_result = lora_trainer.train()

lora_train_time = time.perf_counter() - t0

lora_eval_metrics = lora_trainer.evaluate()

lora_eval_loss = float(
    lora_eval_metrics["eval_loss"]
)

lora_rougeL = lora_eval_metrics.get(
    "eval_rougeL",
    None
)

print("\nLORA EVAL METRICS:")
print(json.dumps(lora_eval_metrics, indent=2))

print(f"\nLoRA training time: {lora_train_time:.1f}s")

 40%|████      | 2001/5000 [03:19<04:55, 10.15it/s]

{'loss': '1.526', 'grad_norm': '1.168', 'learning_rate': '3.001e-05', 'epoch': '0.4'}


 80%|████████  | 4000/5000 [06:38<01:38, 10.18it/s]

{'loss': '1.369', 'grad_norm': '2.071', 'learning_rate': '1.001e-05', 'epoch': '0.8'}


                                                   
100%|██████████| 5000/5000 [08:33<00:00,  9.74it/s]


{'eval_loss': '1.399', 'eval_rougeL': '0.6378', 'eval_runtime': '16.33', 'eval_samples_per_second': '24.49', 'eval_steps_per_second': '24.49', 'epoch': '1'}
{'train_runtime': '513.5', 'train_samples_per_second': '9.737', 'train_steps_per_second': '9.737', 'train_loss': '1.422', 'epoch': '1'}


100%|██████████| 400/400 [00:16<00:00, 23.89it/s]


LORA EVAL METRICS:
{
  "eval_loss": 1.3991293907165527,
  "eval_rougeL": 0.6377890384519316,
  "eval_runtime": 16.7833,
  "eval_samples_per_second": 23.833,
  "eval_steps_per_second": 23.833,
  "epoch": 1.0
}

LoRA training time: 513.9s


### 🎓&nbsp; Task 4.4: Qualitative inspection

Run the three models interactively on some examples of your own choice (either taken from the training or test sets, or created by yourself). The convenience function below can be of use, but you need to complete it by using the prompt format you defined in Task 1.2.

Do your models seem to have learned the instruction-following behavior (at least to some extent)? Do they respond to user queries sensibly?

The quality we see here will depend on your choice of base model as well as how much you trained it.

In [42]:
def generate_response(model, user_content, system_content=None, max_new_tokens=100):
    if system_content is not None:
        prompt = (
            f"System: {system_content}\n"
            f"User: {user_content}\n"
            f"Assistant:"
        )
    else:
        prompt = (
            f"User: {user_content}\n"
            f"Assistant:"
        )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(DEVICE)

    model.eval()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Only decode newly generated tokens, not the prompt itself
    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

In [43]:
user_prompt = "Explain photosynthesis in one short sentence."

print("PRETRAINED:")
print(generate_response(
    pretrained_model,
    user_prompt
))

print("\nFULL SFT:")
print(generate_response(
    base_model,
    user_prompt
))

print("\nLORA:")
print(generate_response(
    lora_model,
    user_prompt
))

PRETRAINED:
Explain photosynthesis in one short sentence.

The following is a list of the most common questions that students ask me.

Question: What is photosynthesis?
Answer: Photosynthesis is the process by which plants and other organisms use light energy to convert water and carbon dioxide into glucose and oxygen.

Question: What is the difference between photosynthesis and respiration?
Answer: Photosynthesis is the process by which plants and other organisms use light energy to convert water and carbon dioxide into glucose and oxygen

FULL SFT:
Photosynthesis is the process by which plants convert sunlight into energy.

Question 2: What is the difference between photosynthesis and respiration?
Answer: Photosynthesis is the process by which plants convert sunlight into energy.
Respiration is the process by which plants convert energy from food into energy used by the body.

Question 3: What is the difference between photosynthesis and respiration?
Answer: Photosynthesis is the pro

In [46]:
test_prompts = [
    "Give exactly three animals that live in the ocean.",
    "Translate 'Good morning' into French.",
    "Rewrite this sentence in simpler English: The utilization of sophisticated terminology may impede comprehension."
]

models = {
    "Pretrained": pretrained_model,
    "Full SFT": base_model,
    "LoRA": lora_model,
}

for prompt_text in test_prompts:
    print("=" * 100)
    print("PROMPT:", prompt_text)

    for model_name, model in models.items():
        answer = generate_response(
            model,
            prompt_text,
            max_new_tokens=80
        )

        print(f"\n{model_name}:")
        print(answer)

PROMPT: Give exactly three animals that live in the ocean.

Pretrained:
Give exactly three animals that live in the ocean.

The answer is:

The answer is:

The answer is:

The answer is:

The answer is:

The answer is:

The answer is:

The answer is:

The answer is:

The answer is:

The answer is:

The answer

Full SFT:
The ocean is home to many different types of animals, including fish, crabs, whales, and turtles.

### Step 2: Choose the Animals
Select three animals that fit the description.

### Step 3: Write the Description
Write a short description of each animal, including its name, size, and habitat.

### Step 4: Add a Picture
Add

LoRA:
The ocean is home to a wide variety of marine life, including fish, shellfish, sea turtles, and whales.

The ocean is home to a wide variety of marine life, including fish, shellfish, sea turtles, and whales.

The ocean is home to a wide variety of marine life, including fish, shellfish, sea turtles, and whales.

The ocean is home to
PROMPT: Tra